# Backpropagation Conv

1. We'll recieve the gradient of $\frac{\partial L} {\partial Y}$. This again is the gradient of the loss w.r.t. output of the convolution layer. We now would like to create 
  * $\frac{\partial L} {\partial X}$ : gradient w.r.t. the input (X)
  * $\frac{\partial L} {\partial W}$ : gradient w.r.t. the input (W)
  * $\frac{\partial L} {\partial b}$ : gradient w.r.t. the input (b)

2. Pad inputs: 
  As we applied pading during the forward pass, we have to apply this same padding to our dvalues to ensure alignment during convolution.
3. Iterate over the output spatial dimensions:
   For each training sample S, output channel C_out, and spatial location (H, W):
  *  Compute the region of the input that contributed to that output at (i, j): (our sliding window)
  * Use the upstream gradient at (S, H, W, C_out) to:
    * Update $\frac{\partial L} {\partial W}$:
      * Multiply the input slice by the gradient scaler and accumulate to $\frac{\partial L} {\partial W[c_{out}]}$
    * Update $\frac{\partial L} {\partial b}$:
      * Accumulate the gradient scalar into $\frac{\partial L} {\partial b[c_{out}]}$ :
    * Update $\frac{\partial L} {\partial X}$:
      * Multiply the filter C_out with the gradient scaler and add it to the corresponding region in $\frac{\partial L} {\partial X}$
4. Remove padding from $\frac{\partial L} {\partial X}$:
5. Return Gradients: 

1. We'll recieve the gradient of $\frac{\partial L} {\partial Y}$. This again is the gradient of the loss w.r.t. output of the convolution layer. We now would like to create 
  * $\frac{\partial L} {\partial X}$ : gradient w.r.t. the input (X)
  * $\frac{\partial L} {\partial W}$ : gradient w.r.t. the input (W)
  * $\frac{\partial L} {\partial b}$ : gradient w.r.t. the input (b)
This step is quite simple as we'll first create these 

```python
    self.dweights = np.zeros_like(self.filter_weights)  # is filter_weights right? what is shape
    self.dbiases = np.zeros_like(self.biases)    # shape(C_out,) add a scaler to this 
    self.dinputs = np.zeros_like(self.inputs)    # shape(S, H_in, W_in, C_in) 
```

2. Pad inputs: 
  As we applied pading during the forward pass, we have to apply this same padding to our dvalues to ensure alignment during convolution.
During this step, we would like to add padding back to our gradient. 
* Each value in dvalues[i, j] contributes to a window in dinputs which would be the same size as the filter. 
* The gradient from dvalues[i, j] is spread across the corrseponding region in dinputs, weighted by the filter and used during the foward pass
$$ \text{dvalues} = 
\begin{bmatrix}
 1 & 2 & 3 & 0 \\
 0 & 1 & 2 & 3 \\
 3 & 0 & 1 & 2 \\
 2 & 3 & 0 & 1 \\
 \end{bmatrix}
$$

$$ \text{then padded dinputs} = 
\begin{bmatrix}
 0 & 0 & 0 & 0 & 0 & 0 \\
 0 & 0 & 0 & 0 & 0 & 0 \\
 0 & 0 & 0 & 0 & 0 & 0 \\
 0 & 0 & 0 & 0 & 0 & 0 \\
 0 & 0 & 0 & 0 & 0 & 0 \\
 0 & 0 & 0 & 0 & 0 & 0 
 \end{bmatrix}
$$

3. Iterate over the output spatial dimensions:
   For each training sample S, output channel C_out, and spatial location (H, W):
  *  Compute the region of the input that contributed to that output at (i, j): (our sliding window)
  * Use the upstream gradient at (S, H, W, C_out) to:
    * Update $\frac{\partial L} {\partial W}$:
      * Multiply the input slice by the gradient scaler and accumulate to $\frac{\partial L} {\partial W[c_{out}]}$
    * Update $\frac{\partial L} {\partial b}$:
      * Accumulate the gradient scalar into $\frac{\partial L} {\partial b[c_{out}]}$ :
    * Update $\frac{\partial L} {\partial X}$:
      * Multiply the filter C_out with the gradient scaler and add it to the corresponding region in $\frac{\partial L} {\partial X}$
$$
\text{Say we had filters} =
\begin {bmatrix}
 1 & 1 & 1 \\
 1 & 1 & 1 \\
 1 & 1 & 1 
 \end{bmatrix}  
$$

At the first iteration, we are to multiply whatever is at the sliding window padded_dinputs[0:3, 0:3] += filter_weights * dvalues[0,0]
$$ \text{then padded dinputs} = 
\begin{bmatrix}
 1 & 1 & 1 & 0 & 0 & 0 \\
 1 & 1 & 1 & 0 & 0 & 0 \\
 1 & 1 & 1 & 0 & 0 & 0 \\
 0 & 0 & 0 & 0 & 0 & 0 \\
 0 & 0 & 0 & 0 & 0 & 0 \\
 0 & 0 & 0 & 0 & 0 & 0 
 \end{bmatrix}
$$

At the second iteration, padded_dinputs[0:3, 1:4] += filter_weights * dvalues[0, 1]

$$ \text{then padded dinputs} = 
\begin{bmatrix}
 1 & 3 & 3 & 2 & 0 & 0 \\
 1 & 3 & 3 & 2 & 0 & 0 \\
 1 & 3 & 3 & 2 & 0 & 0 \\
 0 & 0 & 0 & 0 & 0 & 0 \\
 0 & 0 & 0 & 0 & 0 & 0 \\
 0 & 0 & 0 & 0 & 0 & 0 
 \end{bmatrix}
$$  

We would continue this proces many times until we are done with our sliding window, afterwards, even though are padded borders are populated with values we are to truncate them and keep only the original dimensions of the images. With the example used, it would actually become a 4x4 matrix again. 

to update our bias, it is the same process where we take the sum of the sample image and add it to the bias as all pixels will contribute to the bias. 
```python
self.dbiases = np.sum(dvalues, axis = (0, 1, 2)) 
```
to update our weights, 
```python
self.dweights = np.einsum('shwxyc,shwd->xycd', self.patches, dvalues)
```
* patches: (S, H_out, W_out, fH, fW, C_in) - the input window from the forward pass
* dvalues: (S, H_out, W_out, C_out) - what graidents are flowing back (what we passed in)
* Notice how we only keep the dimensions xycd or our (fH, fW, C_in , C_out). That means, for each filter position and channel pair we'll accumulate patches[x, y, c_in] * dvalue[c_out] across all samples and spatial postions. 

We would now like to perform the **reverse convolution**
```python
contributions = np.einsum("shwd, sycd -> shwxyc", dvalues, self.filter_weights)
```
* dvalues: (S, H_out, W_out, C_out) - what graidents are flowing back (what we passed in)
* self.filter_weights: (fH, fW, C_in, C_out) these were the filters from the forward pass
* At each position, we'd like to unpack the gradient by multiplying the filter weights. In essence, we are reversing what the convolution did
* In the end we recieve the same shape as patches. Where, at each output positon, we have (fH, fW, C_in) patch of gradients to send back. 

In the matrix example above, we had to update many values so we'll use as_strided where each contribution will be accumulated to the patches. Once we are done, we retain our 4d tensor shape and have completed our dinputs. If we did use padding before, we include a boolean that will truncate whatever borders we accumulated along with their values. Luckily, NumPy can do this quite easily. The full code is shown below. 

In [17]:
import numpy as np
from numpy.lib.stride_tricks import as_strided
class Conv_Layer:
    def __init__(self, input_shape, num_filters = 1, filter_size = (3, 3), strides = (1, 1), padding = "same"):

        #input_shape has form (batch_size, height, width, channels)
        self.input_shape = input_shape
        self.num_filters = num_filters
        self.filter_size = filter_size
        self.strides = strides
        self.padding = padding 
        self.biases = np.zeros(self.num_filters) 

        #We'll handle two scenarios, the first, where we pass in a (n, n, 1) or grayscale image, and a second
        #where we'll handle a (n, n, 3) or RGB image. 
        input_depth = input_shape[-1]
        n = self.filter_size[0] * self.filter_size[1] * input_depth
        std = np.sqrt(2.0 / n)
        
        #We can now do He initaliztion, we'll sample values from a standard distribution N (0, 1) and multiply it by our
        #std value to get N(0, std) 

        self.filter_weights = np.random.randn(
            filter_size[0],         #height
            filter_size[1],         #width
            input_depth,            #depth 
            num_filters             #number of filters
        )* std

    def forward(self, inputs):
        #Extract Input dimensions

        fH, fW = self.filter_size
        sH, sW = self.strides
        S, H_in, W_in, D_in = inputs.shape
        
        #Creating padding depending on padding = same, or padding = valid
        if self.padding == "same":
            P = (self.filter_size[0] - 1) // 2
        else:            
            P = 0

        #We need integer output dimensions, so cast equations to int
        H_out = int((H_in + 2 * P - self.filter_size[0]) / self.strides[0] + 1)
        W_out = int((W_in + 2 * P - self.filter_size[1]) / self.strides[1] + 1)
        
        #(0, 0) -> don't touch the number of samples in the batch
        #(P, P) -> pad top and bottom pixels by P pixels (axis 1)
        #(P, P) -> pad left and right pixels by P pixels (axis 2)
        #(0, 0) -> don't pad depth. 
        #contstant -> add constant_values for the padded values
        padded_inputs = np.pad(array = inputs, 
                            pad_width = ((0, 0), (P, P), (P, P), (0, 0)),
                            mode = 'constant',
                            constant_values = 0)

        #Create an output tensor of size (batch_size, H_out, W_out, C_out)
        self.output = np.zeros((self.input_shape[0], H_out, W_out, self.num_filters))

        #create our sliding window
        self.patches = as_strided(
            padded_inputs,
            shape=(S, H_out, W_out, fH, fW, D_in),
            strides=(
                padded_inputs.strides[0],       # step between samples
                padded_inputs.strides[1] * sH,  # step down a row
                padded_inputs.strides[2] * sW,  # step across a column
                padded_inputs.strides[1],       # move down 1 row inside patch
                padded_inputs.strides[2],       # move right 1 col inside patch
                padded_inputs.strides[3],       # step across channels
            )
        )

        #Keep the samples, h_out, w_out, and the number of channels out. But, iterate over the patch(x, y) with channels c, and with the number of filters d
        self.output = np.einsum('shwxyc,xycd->shwd', self.patches, self.filter_weights)
        self.output += self.biases.reshape((1, 1, 1, self.num_filters)) 

        self.inputs = inputs
        self.padded_inputs = padded_inputs
        return self.output.copy()
        #save the output tensor using self. for backpropogation

    def backward(self, dvalues):
        #extract dvalues dimensions
        S, H_out, W_out, C_out = dvalues.shape
        fH, fW, C_in, C_out = self.filter_weights.shape
        sH, sW = self.strides

        #dbiases has shape c_out as we intend to add dvalues to each filter. 
        self.dbiases = np.sum(dvalues, axis = (0 , 1, 2)) 
        
        self.dweights = np.einsum("shwxyc, shwd -> xycd", self.patches, dvalues)

        padded_dinputs = np.zeros_like(self.padded_inputs, dtype = np.float32)

        contributions = np.einsum("shwd, xycd -> shwxyc", dvalues, self.filter_weights)

        #We'll use our window feature to add whatever contributions we had, 
        #By using writeable, we'll be able to add stuff. 
        dinput_patches = as_strided(
            padded_dinputs,
            shape = (S, H_out, W_out, fH, fW, C_in),
            strides = (
            padded_dinputs.strides[0],       # step between samples
                padded_dinputs.strides[1] * sH,  # step down a row
                padded_dinputs.strides[2] * sW,  # step across a column
                padded_dinputs.strides[1],       # move down 1 row inside patch
                padded_dinputs.strides[2],       # move right 1 col inside patch
                padded_dinputs.strides[3],       # step across channels
            ),
            writeable=True
        )

        dinput_patches += contributions 

        #truncate our borders 
        if self.padding == "same":
            P = (fH - 1) // 2
            self.dinputs = dinput_patches[:, P:-P, P:-P, :] 
        else:
            self.dinputs = dinput_patches 
        return self.dinputs

class ReLU:
    def forward(self, inputs, training):
        self.inputs = inputs
        self.output = np.maximum(0, inputs) #works elementwise

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs < 0] = 0 



In [3]:
import numpy as np
from numpy.lib.stride_tricks import as_strided
class Pooling: 
    def __init__(self, filter_size = (2, 2), strides = (2, 2),
                  padding = "valid", pooling_type = "max"):
        self.filter_size = filter_size
        self.strides = strides
        self.padding = padding
        self.pooling_type = pooling_type

    def forward(self, inputs):
        #Inputs should be of shape (S, H_in, W_in, C = D_in) 
        if inputs.ndim != 4:
            raise ValueError(f"Expected a 4D tensor, got {inputs.ndim} instead.")
        S, H_in, W_in, C = inputs.shape
        fH, fW = self.filter_size
        sH, sW = self.strides

        padding = self.padding
        if padding == "valid":
            H_out = np.floor((H_in - fH) / sH) + 1
            W_out = np.floor((W_in - fW) / sW) + 1
        
        elif padding == "same":
            pad_h = max((H_out - 1) * sH + fH - H_in, 0)
            pad_w = max((W_out - 1) * sW + fW - W_in, 0)
            pad_top = pad_h // 2
            pad_bottom = pad_h - pad_top
            pad_left = pad_w // 2
            pad_right = pad_w - pad_left
            inputs = np.pad(inputs, ((0,0), (pad_top,pad_bottom), (pad_left,pad_right), (0,0)), mode='constant')
        else: 
            raise ValueError(f"Expected padding == valid or same, recieved {padding} instead")

        #cast our output dimensions into ints from floats. 
        H_out, W_out = int(H_out), int(W_out)

        #create output tensor with new sizes
        self.output = np.zeros(shape = (S, H_out, W_out, C))
        self.inputs = inputs
        patches = as_strided(
            inputs,
            shape = (S, H_out, W_out, fH, fW, C), 
            strides = (
                inputs.strides[0],      #step between samples
                inputs.strides[1] * sH, #step between rows
                inputs.strides[2] * sW, #step between columns
                inputs.strides[1],      #Move down 1 row inside patch
                inputs.strides[2],      #move right 1col inside patch
                inputs.strides[3],      #step between each channel
            )
        )

        if self.pooling_type == "max":
            pooled = patches.max(axis = (3, 4)) 
            #We'll reshape the window to become a 1d array of size fH * fW
            patches_reshaped = patches.reshape(S, H_out, W_out, fH * fW, C)
            flat_indicies = patches_reshaped.argmax(axis = 3)

            #Now, we'll convert those flat indicies back to row col coordinates withing each
            #(fH, fW) patch
            max_rows, max_cols = np.unravel_index(flat_indicies, (fH, fW)) 
            self.max_indicies = (max_rows, max_cols) 
            
        #Store both of these for backprop
        self.inputs = inputs
        self.output = pooled
        return self.output

    def backward(self, dvalues):
        
        #We want the same shape as self.inputs, we'll populate the tensor with zeros at first then unpool later.
        self.dinputs = np.zeros_like(self.inputs)
        S, H_out, W_out, C = dvalues.shape
        fH, fW = self.filter_size
        sH, sW = self.strides
        max_rows, max_cols = self.max_indicies
        
        s_idx = np.arange(S)[:, None, None, None]      # Shape: (S, 1, 1, 1)
        h_idx = np.arange(H_out)[None, :, None, None]  # Shape: (1, H_out, 1, 1)
        w_idx = np.arange(W_out)[None, None, :, None]  # Shape: (1, 1, W_out, 1)
        c_idx = np.arange(C)[None, None, None, :]      # Shape: (1, 1, 1, C)
        
        # Calculate where in the input each gradient should go
        # Broadcasting creates arrays of shape (S, H_out, W_out, C)
        input_h = h_idx * sH + max_rows  # h_idx broadcasts, max_rows is already (S, H_out, W_out, C)
        input_w = w_idx * sW + max_cols
        
        # Accumulate gradients at the right positions
        # np.add.at handles if multiple output positions map to same input position
        np.add.at(self.dinputs, (s_idx, input_h, input_w, c_idx), dvalues)
        return self.dinputs



In [23]:
import numpy as np
data = np.load("../CodeTest/fashion_mnist_train.npz")
X, y = data["X"], data["y"]

print(X.shape) #60000 samples, by a 28 by 28 grid. 
X = X[..., np.newaxis]
print(X.shape) #60000 samples, by a 28 by 28 grid, with channel = 1 for grayscale. 
sample_batch = X[:128]
print(sample_batch.shape) #128 samples, lets pass this into our convolution layer. 

#sample_batch.shape[1:] = (28, 28, 1)
conv = Conv_Layer(input_shape = sample_batch.shape[1:], num_filters = 2, filter_size = (3, 3), strides = (1, 1), padding = "same") 
output = conv.forward(inputs = sample_batch)


activation = ReLU()
activation.forward(inputs = output, training = 0) #training is dummy paramter

pooling = Pooling(filter_size = (2, 2), strides = (2, 2),
                  padding = "valid", pooling_type = "max")
pooling_forward_output = pooling.forward(inputs = activation.output)

print(conv.output.shape)
print(activation.output.shape)
print(pooling.output.shape)

print("Now let's do the backwards pass")

pooling.backward(dvalues = pooling_forward_output)
activation.backward(dvalues = pooling.dinputs)
conv.backward(dvalues = activation.dinputs)

print(pooling.dinputs.shape)
print(activation.dinputs.shape)
print(conv.dinputs.shape)

(60000, 28, 28)
(60000, 28, 28, 1)
(128, 28, 28, 1)
(128, 28, 28, 2)
(128, 28, 28, 2)
(128, 14, 14, 2)
Now let's do the backwards pass
(128, 28, 28, 2)
(128, 28, 28, 2)
(128, 26, 26, 3, 3, 1)


In [ ]:
import numpy as np
import cupy as cp
from cupy.lib.stride_tricks import as_strided


scatter_contributions_kernel = cp.ElementwiseKernel(
    in_params='''
        float32 contribution, 
        int32 S, int32 H_out, int32 W_out,
        int32 fH, int32 fW, int32 C_in,
        int32 H_padded, int32 W_padded,
        int32 sH, int32 sW
    ''',
    out_params='raw float32 padded_dinputs',
    operation=r'''
        // i is the linear index into contributions array
        // Decode: (s, h_out, w_out, fh, fw, c_in)
        int c_in = i % C_in;
        int fw = (i / C_in) % fW;
        int fh = (i / (C_in * fW)) % fH;
        int w_out = (i / (C_in * fW * fH)) % W_out;
        int h_out = (i / (C_in * fW * fH * W_out)) % H_out;
        int s = i / (C_in * fW * fH * W_out * H_out);
        
        // Calculate target position in padded_dinputs
        int h_target = h_out * sH + fh;
        int w_target = w_out * sW + fw;
        
        // Calculate linear index in padded_dinputs
        int target_idx = ((s * H_padded + h_target) * W_padded + w_target) * C_in + c_in;
        
        // Atomic add to handle overlaps
        // 'contribution' (singular) is the current element value
        atomicAdd(&padded_dinputs[target_idx], contribution);
    ''',
    name='scatter_contributions'
)

class Conv_Layer:
    def __init__(self, input_shape, num_filters = 1, filter_size = (3, 3), strides = (1, 1), padding = "same"):

        #input_shape has form height, width, channels)
        self.input_shape = input_shape
        self.num_filters = num_filters
        self.filter_size = filter_size
        self.strides = strides
        self.padding = padding 
        self.biases = cp.zeros(self.num_filters, dtype = cp.float32)

        #We'll handle two scenarios, the first, where we pass in a (n, n, 1) or grayscale image, and a second
        #where we'll handle a (n, n, 3) or RGB image. 
        input_depth = input_shape[-1]
        n = self.filter_size[0] * self.filter_size[1] * input_depth
        std = cp.sqrt(cp.float32(2.0 / n))
        
        #We can now do He initaliztion, we'll sample values from a standard distribution N (0, 1) and multiply it by our
        #std value to get N(0, std) 

        self.filter_weights = (cp.random.randn(
            filter_size[0],         #height
            filter_size[1],         #width
            input_depth,            #depth 
            num_filters             #number of filters
        ).astype(cp.float32)* std)

    def forward(self, inputs, training):
        #Extract Input dimensions

        inputs = inputs.astype(cp.float32, copy=False)
        fH, fW = self.filter_size
        sH, sW = self.strides
        S, H_in, W_in, D_in = inputs.shape
        
        #Creating padding depending on padding = same, or padding = valid
        if self.padding == "same":
            P = (self.filter_size[0] - 1) // 2
        else:            
            P = 0

        #We need integer output dimensions, so cast equations to int
        H_out = int((H_in + 2 * P - self.filter_size[0]) / self.strides[0] + 1)
        W_out = int((W_in + 2 * P - self.filter_size[1]) / self.strides[1] + 1)
        
        #(0, 0) -> don't touch the number of samples in the batch
        #(P, P) -> pad top and bottom pixels by P pixels (axis 1)
        #(P, P) -> pad left and right pixels by P pixels (axis 2)
        #(0, 0) -> don't pad depth. 
        #contstant -> add constant_values for the padded values
        padded_inputs = cp.pad(array = inputs, 
                            pad_width = ((0, 0), (P, P), (P, P), (0, 0)),
                            mode = 'constant',
                            constant_values = 0).astype(cp.float32, copy = False)

        #Create an output tensor of size (batch_size, H_out, W_out, C_out)
        self.output = cp.zeros((S, H_out, W_out, self.num_filters), dtype = cp.float32)

        #create our sliding window
        self.patches = as_strided(
            padded_inputs,
            shape=(S, H_out, W_out, fH, fW, D_in),
            strides=(
                padded_inputs.strides[0],       # step between samples
                padded_inputs.strides[1] * sH,  # step down a row
                padded_inputs.strides[2] * sW,  # step across a column
                padded_inputs.strides[1],       # move down 1 row inside patch
                padded_inputs.strides[2],       # move right 1 col inside patch
                padded_inputs.strides[3],       # step across channels
            )
        )

        #Keep the samples, h_out, w_out, and the number of channels out. But, iterate over the patch(x, y) with channels c, and with the number of filters d
        self.output = cp.einsum('shwxyc,xycd->shwd', self.patches, self.filter_weights)
        self.output += self.biases.reshape((1, 1, 1, self.num_filters)) 

        self.inputs = inputs
        self.padded_inputs = padded_inputs
        return self.output
        #save the output tensor using self. for backpropogation

    def backward(self, dvalues):

        #extract dvalues dimensions
        S, H_out, W_out, C_out = dvalues.shape
        fH, fW, C_in, C_out = self.filter_weights.shape
        sH, sW = self.strides
        H_padded, W_padded = self.padded_inputs.shape[1:3]

        #dbiases has shape c_out as we intend to add dvalues to each filter. 
        self.dbiases = cp.sum(dvalues, axis = (0 , 1, 2)) 
        
        self.dweights = cp.einsum("shwxyc, shwd -> xycd", self.patches, dvalues)

        padded_dinputs = cp.zeros_like(self.padded_inputs, dtype = cp.float32)

        contributions = cp.einsum("shwd, xycd -> shwxyc", dvalues, self.filter_weights)

        contributions = contributions.astype(cp.float32)
        scatter_contributions_kernel(
            contributions.ravel(),
            S, H_out, W_out, fH, fW, C_in,
            H_padded, W_padded, sH, sW,
            padded_dinputs.ravel()
        )
        #truncate our borders 
        if self.padding == "same":
            P = (fH - 1) // 2
            self.dinputs = padded_dinputs[:, P:-P, P:-P, :] 
        else:
            self.dinputs = padded_dinputs 
        return self.dinputs


scatter_avg_pooling_kernel = cp.ElementwiseKernel(
    in_params='''
        float32 dval,
        int32 S, int32 H_out, int32 W_out, int32 C,
        int32 H_in, int32 W_in,
        int32 fH, int32 fW,
        int32 sH, int32 sW
    ''',
    out_params='raw float32 dinputs',
    operation=r'''
        // i is the linear index into dvalues
        // Decode: (s, h_out, w_out, c)
        int c = i % C;
        int w_out = (i / C) % W_out;
        int h_out = (i / (C * W_out)) % H_out;
        int s = i / (C * W_out * H_out);
        
        // Gradient to distribute to each position in the pool
        float grad_per_position = dval / (fH * fW);
        
        // Calculate starting position in input
        int h_start = h_out * sH;
        int w_start = w_out * sW;
        
        // Distribute gradient to all positions in this pool window
        for (int fh = 0; fh < fH; fh++) {
            for (int fw = 0; fw < fW; fw++) {
                int h_in = h_start + fh;
                int w_in = w_start + fw;
                
                // Calculate linear index in dinputs
                int target_idx = ((s * H_in + h_in) * W_in + w_in) * C + c;
                
                // Atomic add to handle overlapping windows
                atomicAdd(&dinputs[target_idx], grad_per_position);
            }
        }
    ''',
    name='scatter_avg_pooling'
)

class Pooling: 
    def __init__(self, filter_size = (2, 2), strides = (2, 2),
                  padding = "valid", pooling_type = "max"):
        self.filter_size = filter_size
        self.strides = strides
        self.padding = padding
        self.pooling_type = pooling_type

    def forward(self, inputs, training):
        #Inputs should be of shape (S, H_in, W_in, C = D_in) 
        if inputs.ndim != 4:
            raise ValueError(f"Expected a 4D tensor, got {inputs.ndim} instead.")
        S, H_in, W_in, C = inputs.shape
        fH, fW = self.filter_size
        sH, sW = self.strides

        padding = self.padding
        if padding == "valid":
            H_out = cp.floor((H_in - fH) / sH) + 1
            W_out = cp.floor((W_in - fW) / sW) + 1
        
        elif padding == "same":
            H_out = cp.ceil(H_in / sH)
            W_out = cp.ceil(W_in / sW)
            pad_h = max((H_out - 1) * sH + fH - H_in, 0)
            pad_w = max((W_out - 1) * sW + fW - W_in, 0)
            pad_top = pad_h // 2
            pad_bottom = pad_h - pad_top
            pad_left = pad_w // 2
            pad_right = pad_w - pad_left
            inputs = cp.pad(inputs, ((0,0), (pad_top,pad_bottom), (pad_left,pad_right), (0,0)), mode='constant')
        else: 
            raise ValueError(f"Expected padding == valid or same, recieved {padding} instead")

        #cast our output dimensions into ints from floats. 
        H_out, W_out = int(H_out), int(W_out)

        #create output tensor with new sizes
        self.output = cp.zeros(shape = (S, H_out, W_out, C), dtype = cp.float32)
        self.inputs = inputs
        patches = as_strided(
            inputs,
            shape = (S, H_out, W_out, fH, fW, C), 
            strides = (
                inputs.strides[0],      #step between samples
                inputs.strides[1] * sH, #step between rows
                inputs.strides[2] * sW, #step between columns
                inputs.strides[1],      #Move down 1 row inside patch
                inputs.strides[2],      #move right 1col inside patch
                inputs.strides[3],      #step between each channel
            )
        )

        if self.pooling_type == "max":
            pooled = patches.max(axis = (3, 4)) 
            #We'll reshape the window to become a 1d array of size fH * fW
            patches_reshaped = patches.reshape(S, H_out, W_out, fH * fW, C)
            flat_indicies = patches_reshaped.argmax(axis = 3)

            #Now, we'll convert those flat indicies back to row col coordinates withing each
            #(fH, fW) patch
            max_rows, max_cols = cp.unravel_index(flat_indicies, (fH, fW)) 
            self.max_indicies = (max_rows, max_cols) 
        
        elif self.pooling_type == "average":
            pooled = patches.mean(axis = (3, 4))
            
        #Store both of these for backprop
        self.inputs = inputs
        self.output = pooled
        return self.output

    def backward(self, dvalues):
        
        #We want the same shape as self.inputs, we'll populate the tensor with zeros at first then unpool later.
        self.dinputs = cp.zeros_like(self.inputs, dtype = cp.float32)
        S, H_out, W_out, C = dvalues.shape
        H_in, W_in = self.inputs.shape[1:3]
        fH, fW = self.filter_size
        sH, sW = self.strides
        
        if self.pooling_type == "max":
            max_rows, max_cols = self.max_indicies
            
            s_idx = cp.arange(S)[:, None, None, None]      # Shape: (S, 1, 1, 1)
            h_idx = cp.arange(H_out)[None, :, None, None]  # Shape: (1, H_out, 1, 1)
            w_idx = cp.arange(W_out)[None, None, :, None]  # Shape: (1, 1, W_out, 1)
            c_idx = cp.arange(C)[None, None, None, :]      # Shape: (1, 1, 1, C)
            
            # Calculate where in the input each gradient should go
            # Broadcasting creates arrays of shape (S, H_out, W_out, C)
            input_h = h_idx * sH + max_rows  # h_idx broadcasts, max_rows is already (S, H_out, W_out, C)
            input_w = w_idx * sW + max_cols
            
            # Accumulate gradients at the right positions
            # cp.add.at handles if multiple output positions map to same input position
            cp.add.at(self.dinputs, (s_idx, input_h, input_w, c_idx), dvalues)
        
        elif self.pooling_type == "average":
            
            scatter_avg_pooling_kernel(
                dvalues.ravel(),
                S, H_out, W_out, C,
                H_in, W_in, 
                fH, fW,
                sH, sW,
                self.dinputs.ravel()
            )
        return self.dinputs


class ReLU:
    def forward(self, inputs, training):
        self.inputs = inputs
        self.output = cp.maximum(0, inputs) #works elementwise

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs < 0] = 0 

In [ ]:
import cupy as cp
data = cp.load("../CodeTest/fashion_mnist_train_cupy.npz")
X, y = data["X"], data["y"]

print(X.shape) #60000 samples, by a 28 by 28 grid. 
X = X[..., np.newaxis]
print(X.shape) #60000 samples, by a 28 by 28 grid, with channel = 1 for grayscale. 
sample_batch = X[:128]
print(sample_batch.shape) #128 samples, lets pass this into our convolution layer. 

#sample_batch.shape[1:] = (28, 28, 1)
conv = Conv_Layer(input_shape = sample_batch.shape[1:], num_filters = 2, filter_size = (3, 3), strides = (1, 1), padding = "same") 
output = conv.forward(inputs = sample_batch)


activation = ReLU()
activation.forward(inputs = output, training = 0) #training is dummy paramter

pooling = Pooling(filter_size = (2, 2), strides = (2, 2),
                  padding = "valid", pooling_type = "max")
pooling_forward_output = pooling.forward(inputs = activation.output)

print(conv.output.shape)
print(activation.output.shape)
print(pooling.output.shape)

print("Now let's do the backwards pass")

pooling.backward(dvalues = pooling_forward_output)
activation.backward(dvalues = pooling.dinputs)
conv.backward(dvalues = activation.dinputs)

print(pooling.dinputs.shape)
print(activation.dinputs.shape)
print(conv.dinputs.shape)


(60000, 28, 28)
(60000, 28, 28, 1)
(128, 28, 28, 1)
(128, 28, 28, 2)
(128, 28, 28, 2)
(128, 14, 14, 2)
Now let's do the backwards pass
(128, 28, 28, 2)
(128, 28, 28, 2)
(128, 28, 28, 1)
